# Building the Baseline Model

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from src.data import build_index, make_split, check_split, split_dataframe
from src.dataset import GTSRBDataset

df = build_index(PROJECT_ROOT/'data'/'Train')
df = make_split(df, 5, 42)
check_split(df)

Number of paths: 39209
Number of Tracks: 1307
Number of classes: 43
train: 43/43 classes
val: 43/43 classes

Leak check:
No leaked tracks.

5 least common classes in Validation set:
split  train  val
label            
0          6    1
19         6    1
29         8    1
32         7    1
37         6    1


In [2]:
import torchvision.transforms as transforms

base_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

In [3]:
from PIL import Image

img = Image.open(PROJECT_ROOT/'data'/'Train'/'7'/'00007_00002_00028.png').convert("RGB")
t_image = base_transforms(img)
print(t_image.shape)
print(t_image.dtype)
print(t_image.min())
print(t_image.max())

torch.Size([3, 32, 32])
torch.float32
tensor(-0.7333)
tensor(1.)


In [4]:
Tdf, Vdf = split_dataframe(df)
train_ds = GTSRBDataset(df=Tdf, transform=base_transforms)
val_ds = GTSRBDataset(df=Vdf, transform=base_transforms)

In [5]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}")
print(f"#Labels: {labels.shape}     Type: {labels.dtype}")

Batch shape: torch.Size([64, 3, 32, 32])
#Labels: torch.Size([64])     Type: torch.int64
